# Calculate sectors footprint with Exiobase

**Related publication**

 - Title: Residual biomass to bio-based chemicals and plastics: ex-ante screening methodology for prioritizing high-impact substitutions
 - Authors: [Nicolas LIENART](https://orcid.org/0009-0001-3259-2819), [Thibaut LECOMPTE](https://orcid.org/0000-0001-9237-8454), [Lorie HAMELIN](https://orcid.org/0000-0001-9092-1900) 
 - Journal: Resources, Conservation and Recycling (RCR) - Elsevier
 - Doi: #todo
 - Git repository (Forge INRAE): https://forge.inrae.fr/nicolas.lienart/screen-lca-paper-supplementary-code
 - Git repository (GitHub): https://github.com/nicolnt/screen-lca-paper-supplementary-code

**Description and details**

This code calculates the greenhouse gas (GHG) footprint of France (FR) imports and production for year 2019 based on Exiobase data. The GHG indicator is the GWP100 calculated using coefficients from IPCC AR6 2021. Refer to the aforementioned main manuscript and accompanying supplementary information documents for more details.

**Updates**

 - May 04, 2026: Ready for submission
 - August 08, 2026: Update variable names and clarify some functions

**Package versions**

 - See [`environment.yml`](./environment.yml)
 - pymrio library code fetched directly from GitHub. See: https://github.com/IndEcol/pymrio/issues/144

 - Exiobase version: 3.9.5
   - dataset: IOT_2019_pxp
   - Available at https://doi.org/10.5281/zenodo.14869924

**Relevant references**

 - Pellan, M. et al. (2024) “Integrating Consumption-Based Metrics into Sectoral Carbon Budgets to Enhance Sustainability Monitoring of Building Activities,” Sustainability, 16(16), p. 6762. Available at: https://doi.org/10.3390/su16166762.
 <br>_Provided insiration on the use of Pymrio with Exiobase, see associated Github repository_
 - Aguilar-Hernandez, G.A. (2025) “A Novel Framework to Measure Circularity Trade-offs and Synergies in the Global Context,” Journal of Circular Economy, 3(1). Available at: https://doi.org/10.55845/YTGD9041.
 <br>_Provided the Python code for GHG stressors characterization, see associated Github repository_
 - Andrieu, B. et al. (2024) “An open-access web application to visualise countries’ and regions’ carbon footprints using Sankey diagrams,” Communications Earth & Environment, 5(1), pp. 1–9. Available at: https://doi.org/10.1038/s43247-024-01378-8.
  <br>_Provided insiration on the use of Pymrio with Exiobase, see associated Github repository_
 - Wiedmann, T. (2017) “On the decomposition of total impact multipliers in a supply and use framework,” Journal of Economic Structures, 6(1), p. 11. Available at: https://doi.org/10.1186/s40008-017-0072-0.
  <br>_Provided example of input-output calculations and results_
 - Stadler, K. (2021) “Pymrio – A Python Based Multi-Regional Input-Output Analysis Toolbox,” Journal of Open Research Software, 9(1). Available at: https://doi.org/10.5334/jors.251.
 <br>_Provided example of input-output calculations and results_
 - https://pymrio.readthedocs.io/en/latest/
 <br>_Pymrio library documentation_

## Initialization

In [1]:
import pymrio
import numpy as np
import pandas as pd
from datetime import datetime

In [2]:
from importlib.metadata import version
version('pymrio')

'0.6.3'

In [4]:
OUTPUT_PATH = 'output/'
IO_DATA_PATH = './Exiobase_data/'
EXIOBASE_VERSION = 'EXIOBASE_v3.9.5'
EXIOBASE_MODEL_AND_YEAR = 'IOT_2019_pxp'
EXIOBASE_PATH = IO_DATA_PATH + EXIOBASE_VERSION + '/' + EXIOBASE_MODEL_AND_YEAR

print(EXIOBASE_PATH)

./Exiobase_data/EXIOBASE_v3.9.5/IOT_2019_pxp


In [5]:
# NOTE: Load Exiobase data
# Takes < 1 min
exiobase = pymrio.parse_exiobase3(EXIOBASE_PATH)

In [6]:
# NOTE: Necessary to calculate the L matrix (exiobase.L) and other missing matrices
# Can take a several minutes
exiobase.calc_all()

In [5]:
# NOTE: Export function (optional)
# USAGE: export_csv(exiobase, exiobase.impacts.D_cba, 'name')
def export_csv(exiobase, data, name):
    date = datetime.now().strftime('%Y%m%d')
    export_path = 'output/'
    version = "exiobase_v" + exiobase.meta.system + exiobase.meta.description[len(exiobase.meta.description) - 4:]
    #print(date + " - " + version)
    data.to_csv(export_path + date + ' - output_' + version + '_' + name + '.csv')

## Stressors characterization

In [ ]:
# NOTE: Export S matrix for external characterization
S = exiobase.air_emissions.S
S.to_csv(OUTPUT_PATH+'S.csv')

In [ ]:
# Adapted from https://github.com/aguilarga/circularity_trade-offs-synergies_supplementary_material/blob/main/ghg_calculation_exiobase_v3.9.5.py
# Linked to this publication: Aguilar-Hernandez, G.A. (2025) “A Novel Framework to Measure Circularity Trade-offs and Synergies in the Global Context,” Journal of Circular Economy, 3(1). Available at: https://doi.org/10.55845/YTGD9041.
# Emissions extension
env_ext = pd.read_csv(OUTPUT_PATH+'S.csv', sep=',', index_col=[0], header=[0, 1])  # impacts matrix

# GWP values (IPCC AR6 data in https://zenodo.org/records/6483002)
gwp_values = {
    # Carbon Dioxide (CO2)
    'CO2 - combustion - air': 1,
    'CO2 - agriculture - peat decay - air': 1,
    
    # Methane (CH4)
    'CH4 - combustion - air': 27,
    'CH4 - agriculture - air': 27,
    
    # Nitrous Oxide (N2O)
    'N2O - combustion - air': 273,
    'N2O - agriculture - air': 273,
    
    # Sulfur Hexafluoride (SF6)
    'SF6 - air': 25200,
    
    # Hydrofluorocarbons (HFCs)
    'HFC - air': 1,  # In kg CO2-eq from EXIOBASE
    
    # Perfluorocarbons (PFCs)
    'PFC - air': 1  # In kg CO2-eq from EXIOBASE
}

# Filter relevant emissions and apply GWP values
ghg_emissions = env_ext.loc[env_ext.index.intersection(gwp_values.keys())]  # Select GHGs
ghg_emissions_gwp = ghg_emissions.mul(pd.Series(gwp_values), axis=0)  # Apply GWP values
ghg_gwp_sum = ghg_emissions_gwp.sum()

#ghg_gwp_sum.index = env_ext.columns

ghg_gwp_sum.to_csv(OUTPUT_PATH+"ghg_emissions_extension.csv")

In [18]:
# NOTE: Build M matrix from the externally characterized GWP impacts
GHG_extension = pd.read_csv(OUTPUT_PATH+'ghg_emissions_extension.csv', index_col=[0,1])

# NOTE: Diagonal version
diag_GHG_extension = pd.DataFrame(np.diag(GHG_extension['0']))
diag_GHG_extension.index = GHG_extension['0'].index
diag_GHG_extension.columns = GHG_extension['0'].index
diag_M_GHG = diag_GHG_extension.dot(exiobase.L)

# NOTE: Series version
M_GHG = GHG_extension['0'].dot(exiobase.L)
M_GHG

region  sector                                           
AT      Paddy rice                                           0.000000e+00
        Wheat                                                1.098467e+06
        Cereal grains nec                                    1.587213e+06
        Vegetables, fruit, nuts                              4.916824e+05
        Oil seeds                                            5.579914e+05
                                                                 ...     
WM      Membership organisation services n.e.c. (91)         4.579774e+05
        Recreational, cultural and sporting services (92)    3.916040e+05
        Other services (93)                                  9.897590e+05
        Private households with employed persons (95)        4.908523e+05
        Extra-territorial organizations and bodies           0.000000e+00
Name: 0, Length: 9800, dtype: float64

## Accounting approach: Footprint of production + imports (as used in the main paper case study)

### 1.1 Imported final demand footprint in France per product

In [ ]:
Y_FR_imports = exiobase.Y.loc[:, ['FR']]

Y_FR_imports.loc[['FR'], ['FR']] = 0

region                                                                                     FR  \
category                                          Final consumption expenditure by households   
sector                                                                                          
Paddy rice                                                                                0.0   
Wheat                                                                                     0.0   
Cereal grains nec                                                                         0.0   
Vegetables, fruit, nuts                                                                   0.0   
Oil seeds                                                                                 0.0   
...                                                                                       ...   
Membership organisation services n.e.c. (91)                                              0.0   
Recreational, cultural and sporting services (92)                                         0.0   
Other services (93)                                                                       0.0   
Private households with employed persons (95)                                             0.0   
Extra-territorial organizations and bodies                                                0.0   

region                                                                                                                                  \
category                                          Final consumption expenditure by non-profit organisations serving households (NPISH)   
sector                                                                                                                                   
Paddy rice                                                                                       0.0                                     
Wheat                                                                                            0.0                                     
Cereal grains nec                                                                                0.0                                     
Vegetables, fruit, nuts                                                                          0.0                                     
Oil seeds                                                                                        0.0                                     
...                                                                                              ...                                     
Membership organisation services n.e.c. (91)                                                     0.0                                     
Recreational, cultural and sporting services (92)                                                0.0                                     
Other services (93)                                                                              0.0                                     
Private households with employed persons (95)                                                    0.0                                     
Extra-territorial organizations and bodies                                                       0.0                                     

region                                                                                         \
category                                          Final consumption expenditure by government   
sector                                                                                          
Paddy rice                                                                                0.0   
Wheat                                                                                     0.0   
Cereal grains nec                                                                         0.0   
Vegetables, fruit, nuts                                                                   0.0   
Oil seeds                    

In [16]:
Y_FR_imports = exiobase.Y.loc[:, ['FR']]
#specific_index_columns

# NOTE: Setting FR final demand satisfied by FR to 0. Avoiding double counting with FR production footprint
Y_FR_imports.loc[['FR'], ['FR']] = 0

# NOTE: Join the different final demand category columns together, no distinction is made
Y_FR_imports = Y_FR_imports.sum(1)

diag_Y_FR_imports = pd.DataFrame(np.diag(Y_FR_imports))
diag_Y_FR_imports.index = Y_FR_imports.index 
diag_Y_FR_imports.columns = Y_FR_imports.index

In [19]:
# NOTE: Multiply FR final demand with the footprint data
Y_FR_imports_footprint = M_GHG.dot(diag_Y_FR_imports)
Y_FR_imports_footprint

region  sector                                           
AT      Paddy rice                                           0.000000e+00
        Wheat                                                9.424429e+05
        Cereal grains nec                                    2.347156e+05
        Vegetables, fruit, nuts                              3.406475e+06
        Oil seeds                                            1.192398e+05
                                                                 ...     
WM      Membership organisation services n.e.c. (91)         1.402394e+04
        Recreational, cultural and sporting services (92)    1.384434e+07
        Other services (93)                                  1.095734e+07
        Private households with employed persons (95)        1.252446e-01
        Extra-territorial organizations and bodies           0.000000e+00
Name: 0, Length: 9800, dtype: float64

In [20]:
Y_FR_imports_footprint_agg = Y_FR_imports_footprint.groupby('sector').sum()
Y_FR_imports_footprint_agg

sector
Additives/Blending Components                                                                   4.415480e-04
Air transport services (62)                                                                     1.811409e+10
Aluminium and aluminium products                                                                1.877043e+07
Aluminium ores and concentrates                                                                 6.926591e+04
Animal products nec                                                                             2.494577e+08
                                                                                                    ...     
Wood material for treatment, Re-processing of secondary wood material into new wood material    0.000000e+00
Wood waste for treatment: incineration                                                          0.000000e+00
Wood waste for treatment: landfill                                                              0.000000e+00
Wool, silk-w

In [11]:
# NOTE: Export restults to CSV
# export_csv(exiobase, Y_FR_imports_footprint_agg, "Y_FR_imports_footprint")

### 1.2 Imported intermediate demand footprint in France per product

In [ ]:
# NOTE: Only conserve intermediate industry demand from FR
Z_FR_imports = exiobase.Z.loc[:, ['FR']]

# NOTE: Sum all intermediate demand categories (columns) together,
# no distinction is made regarding the origin of the industry
Z_FR_imports = Z_FR_imports.sum(1)

# NOTE: Set locally satisfied intermediate demand from FR to 0.
# Avoiding double counting with FR production footprint
Z_FR_imports.loc['FR'] = 0

diag_Z_FR_imports = pd.DataFrame(np.diag(Z_FR_imports))
diag_Z_FR_imports.index = Z_FR_imports.index 
diag_Z_FR_imports.columns = Z_FR_imports.index
#diag_Z_FR

region  sector                                           
AT      Paddy rice                                           0.000000e+00
        Wheat                                                6.194457e+06
        Cereal grains nec                                    8.594080e+07
        Vegetables, fruit, nuts                              3.388182e+05
        Oil seeds                                            2.473795e+06
                                                                 ...     
WM      Membership organisation services n.e.c. (91)         5.172638e+06
        Recreational, cultural and sporting services (92)    1.034490e+07
        Other services (93)                                  1.416065e+07
        Private households with employed persons (95)        1.302797e+09
        Extra-territorial organizations and bodies           0.000000e+00
Name: 0, Length: 9800, dtype: float64

In [ ]:
# NOTE: Multiply FR intermediate demand with the footprint data

# For EXIOBASE v3.9.5
Z_FR_imports_footprint = M_GHG.dot(diag_Z_FR_imports)

# For EXIOBASE v3.8.2
# Z_FR_imports_footprint = exiobase.impacts.M.dot(diag_Z_FR_imports)

Z_FR_imports_footprint

In [16]:
Z_FR_imports_footprint_agg = Z_FR_imports_footprint.groupby('sector').sum()
Z_FR_imports_footprint_agg

sector
Additives/Blending Components                                                                   1.259082e+08
Air transport services (62)                                                                     2.832577e+09
Aluminium and aluminium products                                                                5.356507e+09
Aluminium ores and concentrates                                                                 7.356838e+07
Animal products nec                                                                             1.289176e+07
                                                                                                    ...     
Wood material for treatment, Re-processing of secondary wood material into new wood material    0.000000e+00
Wood waste for treatment: incineration                                                          5.952595e+06
Wood waste for treatment: landfill                                                              3.318530e+06
Wool, silk-w

In [14]:
#export_csv(exiobase, Z_FR_imports_footprint_agg, "Z_FR_imports_footprint")

### 2.1 Production footprint in France per product, for intermediate demand (FR and exports)

In [22]:
Z_FR_agg = exiobase.Z.copy()

# NOTE: all other producers in Z than FR are set to 0, we only consider production happening in FR
Z_FR_agg.loc[Z_FR_agg.index.get_level_values('region') != 'FR'] = 0


# NOTE: Only conserve industry production from FR.
# Dimensions: 2D (product) x 2D (region x product) | 200 x 9800
# Z_FR = exiobase.Z.loc[[('FR')]]
Z_FR_agg

# NOTE: Sum all output categories (columns) together,
# No distinction is made regarding the demanding industry
# Dimensions: 2D (product) x 0 | 200
Z_FR_agg = Z_FR_agg.sum(1)
Z_FR_agg

# NOTE: Diagonalized Z (gross output) where only FR output is kept
# Dimensions: 2D (region x product) x 2D (region x product) | 9800 x 9800
diag_Z_FR_agg = pd.DataFrame(np.diag(Z_FR_agg))
diag_Z_FR_agg.index = Z_FR_agg.index 
diag_Z_FR_agg.columns = Z_FR_agg.index
# diag_Z_FR_agg

In [ ]:
# NOTE: Multiply FR production with the footprint data.
# Dimensions: 1D (indicator) x 2D (region x product) | 126 x 9800

# For EXIOBASE v3.8.2
# Z_FR_production_footprint = exiobase.impacts.M.dot(diag_Z_FR)

# For EXIOBASE v3.9.5
Z_FR_production_footprint = M_GHG.dot(diag_Z_FR_agg)

# Dimensions: 1D (indicator) x 1D (product) | 126 x 200
Z_FR_production_footprint

In [18]:
Z_FR_production_footprint_agg = Z_FR_production_footprint.groupby('sector').sum()
Z_FR_production_footprint_agg

sector
Additives/Blending Components                                                                   1.371827e+08
Air transport services (62)                                                                     1.234101e+10
Aluminium and aluminium products                                                                2.231743e+09
Aluminium ores and concentrates                                                                 1.729967e+07
Animal products nec                                                                             5.969149e+08
                                                                                                    ...     
Wood material for treatment, Re-processing of secondary wood material into new wood material    0.000000e+00
Wood waste for treatment: incineration                                                          1.639373e+08
Wood waste for treatment: landfill                                                              1.533510e+08
Wool, silk-w

In [17]:
# export_csv(exiobase, Z_FR_production_footprint_agg, "Z_FR_production_footprint")

### 2.2 Production footprint in France per product, for final demand (FR and exports)

In [23]:
Y_FR_production = exiobase.Y.copy()

# NOTE: all other producers in Z than FR are set to 0, we only consider production happening in FR
Y_FR_production.loc[Y_FR_production.index.get_level_values('region') != 'FR'] = 0

Y_FR_production

# NOTE: Join the different final demand category columns together, no distinction is made
Y_FR_production = Y_FR_production.sum(1)


diag_Y_FR_production = pd.DataFrame(np.diag(Y_FR_production))
diag_Y_FR_production.index = Y_FR_production.index 
diag_Y_FR_production.columns = Y_FR_production.index

In [25]:
# NOTE: Multiply FR production feeding different final demand with the footprint data

# For EXIOBASE v3.9.5
Y_FR_production_footprint = M_GHG.dot(diag_Y_FR_production)

# For EXIOBASE v3.8.2
# Y_FR_production_footprint exiobase.impacts.M.dot(diag_Y_FR_production)

In [26]:
Y_FR_production_footprint_agg = Y_FR_production_footprint.groupby('sector').sum()
Y_FR_production_footprint_agg

sector
Additives/Blending Components                                                                   3.931553e+05
Air transport services (62)                                                                     1.504966e+10
Aluminium and aluminium products                                                                8.032658e+07
Aluminium ores and concentrates                                                                 5.152921e+04
Animal products nec                                                                             1.594196e+09
                                                                                                    ...     
Wood material for treatment, Re-processing of secondary wood material into new wood material    0.000000e+00
Wood waste for treatment: incineration                                                          3.023014e+07
Wood waste for treatment: landfill                                                              3.408543e+07
Wool, silk-w

In [20]:
# export_csv(exiobase, Y_FR_production_footprint_agg, "Y_FR_production_footprint")